# **Database Migration from SQL Server from PostgreSQL**

In [ ]:
import os
import pandas as pd 
import pyodbc
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.exc import SQLAlchemyError
import urllib

### 1. Load Credentials

In [2]:
load_dotenv()

True

In [3]:
sql_host = os.getenv("SQL_SERVER_HOST")
sql_db = os.getenv("SQL_SERVER_DB")
sql_user = os.getenv("SQL_SERVER_USER")
sql_password = os.getenv("SQL_SERVER_PASSWORD")

# PostgreSQL
pg_host = os.getenv("POSTGRES_HOST")
pg_port = os.getenv("POSTGRES_PORT")
pg_db = os.getenv("POSTGRES_DB")
pg_user = os.getenv("POSTGRES_USER")
pg_password = os.getenv("POSTGRES_PASSWORD")

# ODBC Driver
driver = '{ODBC Driver 17 for SQL Server}'

### 2. Connect to SQL Server

In [4]:
print("Connecting to SQL Server ...")
print(f"     Server: {sql_host}")
print(f"     Database: {sql_db}")

Connecting to SQL Server ...
     Server: 127.0.0.1,1433
     Database: AdventureWorks2025


In [ ]:
# Build connection string for SQLAlchemy
try: 
    sql_connection_string_sqlalchemy = urllib.parse.quote_plus(
        f'DRIVER={driver};'
        f'SERVER={sql_host};'
        f'DATABASE={sql_db};'
        f'UID={sql_user};'
        f'PWD={sql_password};'
        f'Encrypt=yes;'
        f'TrustServerCertificate=yes;'
    )
    engine = create_engine(f"mssql+pyodbc:///?odbc_connect={sql_connection_string_sqlalchemy}")
    print("[SUCCESS] -> Connection to SQL Server now live.")
except SQLAlchemyError as e:
    print(f"SQL Server connection failed: {e}")
    print(""" How to troubleshoot:
            > 1. Check credentials in .env file are correct
            > 2. Verify SQL Server is running
            > 3. Check Windows Authentification is enable or SQL Login on macOS
            > 4. Check that Encryption is Madatory for connection            
          """)

[SUCCESS] -> Connection to SQL Server now live.


### 3. Connect to PostgreSQL

In [6]:
print("Connecting to PostgreSQL ...")
print(f"     Server: {pg_host}")
print(f"     Database: {pg_db}")

Connecting to PostgreSQL ...
     Server: localhost
     Database: AdventureWorks2025_UAT


In [7]:
try: 
    pg_connection = psycopg2.connect(
        host = pg_host,
        port = pg_port,
        database = pg_db,
        user = pg_user,
        password = pg_password    
    )
    pg_cursor = pg_connection.cursor()
    pg_cursor.execute("SELECT version();")
    pg_version = pg_cursor.fetchone()[0]
    print("Connected to PostgreSQL")
    print(f"     Version: {pg_version[:50]}... \n")
    
except psycopg2.OperationalError as e:
    print(f"PostgreSQL connection failed: {e}")
    print(""" How to troubleshoot:
          > 1. Check PostgreSQL is running
          > 2. Verify username and password
          > 3. Check database exists
          """)
    
except Exception as e: 
    print(f"Unexpected errors: {e}")
    raise

Connected to PostgreSQL
     Version: PostgreSQL 18.3 (Homebrew) on aarch64-apple-darwin... 



### 4. Migration Dependency Analysis

#### 4.1 Database Schema Exploration
